In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 📰 GenAI News Digest Assistant

---

## Problem  
In today’s world, people are overwhelmed with information. It’s hard to keep up with trustworthy and relevant news.

## Solution  
This GenAI-powered assistant fetches real-time news, summarizes key points using LLMs, and presents results in a clean, structured format.

## GenAI Capabilities Used  
- ✅ Few-shot Prompting  
- ✅ Retrieval-Augmented Generation (RAG) [simulated using live article fetch + LLM]  
- ✅ Structured Output (JSON format)


## 🔽 Step 1: Set Up & Install Libraries


In [2]:
!pip install openai newspaper3k --quiet


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 48.6 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.3/106.3 kB 4.7 MB/s eta 0:00:00


## 🔽 Step 2: Fetch Latest News Headlines


In [3]:
import requests

API_KEY = "b91a249b04834cb4a505357db4ca32b0"  
url = ('https://newsapi.org/v2/top-headlines?country=us&category=technology&apiKey=' + API_KEY)

response = requests.get(url)
articles = response.json().get('articles', [])

for a in articles[:3]:
    print(a['title'], "\n", a['url'], "\n")


Microsoft's Copilot Vision is now free for all Edge users - here's how it works - ZDNet 
 https://www.zdnet.com/article/microsofts-copilot-vision-is-now-free-for-all-edge-users-heres-how-it-works/ 

The Ultimate Nintendo Switch 2 Q&A: We answer more than 90 of your Switch 2 questions - Video Games Chronicle 
 https://www.videogameschronicle.com/features/the-ultimate-nintendo-switch-2-qa-we-answer-more-than-90-of-your-switch-2-questions/ 

Apple One could add a new service in iOS 19, here’s what’s coming - 9to5Mac 
 https://9to5mac.com/2025/04/18/apple-one-could-add-a-new-service-in-ios-19-heres-whats-coming/ 



## 🔽 Step 3: Extract Full Article Content


In [10]:
!pip install "lxml[html_clean]" newspaper3k


In [11]:
from newspaper import Article

def extract_article(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        return article.text
    except:
        return "Error"

# Test
article_text = extract_article(articles[0]['url'])
print(article_text[:1000])


Lance Whitney / Elyse Betters Picaro / ZDNET

Looking for an AI that can analyze, summarize, and answer questions about your current web page? You may want to try Microsoft's Copilot Vision feature.

In a Bluesky post from Wednesday, Mustafa Suleyman, CEO of Microsoft AI, announced the official debut of Copilot Vision for all Edge users. Initially rolled out last October as an experimental feature, Copilot Vision expanded its reach in December as a preview, but only for Copilot Pro subscribers. Now, the feature is free for any Edge user with a Microsoft account.

Also: With Copilot Studio's new skill, your AI agent can use websites and apps just like you do

"Copilot Vision is out now, free in Edge," Suleyman said in his post. "It can literally see what you see on screen (if you opt in). Pretty amazing! It'll think out loud with you when you're browsing online. No more over-explaining, copy-pasting, or struggling to put something into words."

What's so special about Copilot Vision?

U

In [14]:
from newspaper import Article

def extract_article(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        return article.text
    except Exception as e:
        return f"Error: {str(e)}"

# Test with a valid URL
valid_url = "https://realnewswebsite.com/article"  # Replace with an actual URL
article_text = extract_article(valid_url)
print(article_text[:1000])


## 🔽 Step 4: Summarize Articles Using LLM (Offline BART Model)


In [16]:
from transformers import pipeline

# Load the BART summarizer pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Test with an article (replace with your extracted article text)
article_text = """Your article text here"""

# Summarize the article
summary = summarizer(article_text, max_length=100, min_length=50, do_sample=False)

# Print the summary
print(summary[0]['summary_text'])


Device set to use cpu
Your max_length is set to 100, but your input_length is only 6. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)


CNN.com will feature iReporter photos in a weekly Travel Snapshots gallery. Please submit your best shots of the U.S. for next week. Visit CNN.com/Travel each week for a new gallery of snapshots. Visit http://www.dailymail.co.uk/travel/features/trending-trends-in-the-U.S.-next-week.


In [17]:
!pip install faiss-cpu --quiet


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 46.0 MB/s eta 0:00:00:00:0100:01


In [18]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def generate_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

# Example summary
summary = "This is a sample summary of the article content."

# Generate embedding
embedding = generate_embedding(summary)


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

## 🔽 Step 5: Embed Summaries with FAISS (for RAG-style Search)


In [20]:
import faiss
import numpy as np

# Create a FAISS index
dimension = 384  # Use the dimension of the embeddings
# Build FAISS index for quick vector search of summaries
index = faiss.IndexFlatL2(dimension)  # L2 distance metric

# Convert embeddings to numpy array
embeddings_array = np.array([embedding], dtype=np.float32)

# Add embeddings to FAISS index
index.add(embeddings_array)


In [21]:
def query_index(query_text):
    query_embedding = generate_embedding(query_text)
    query_embedding = np.array([query_embedding], dtype=np.float32)
    distances, indices = index.search(query_embedding, k=1)
    return distances, indices

# Query example
query = "What is the latest news on technology?"
distances, indices = query_index(query)
print("Distances:", distances)
print("Indices:", indices)


Distances: [[38.590015]]
Indices: [[0]]


## 🔽 Step 6: Define Retrieval Function


In [22]:
def query_news(query_text, index, k=1):
    # Generate embedding for the query
    query_embedding = generate_embedding(query_text)
    query_embedding = np.array([query_embedding], dtype=np.float32)
    
    # Search for the most similar summary
    distances, indices = index.search(query_embedding, k)
    
    # Retrieve the corresponding summary
    summary_index = indices[0][0]
    return summaries[summary_index]  # Replace 'summaries' with your list of stored summaries


## 🔽 Step 7: Chatbot Function (Ask Questions About the News)


In [23]:
# Simulate a chatbot that answers based on relevant summaries
def chatbot(query):
    # Find the most similar news summary to user's query
    relevant_summary = query_news(query, index)
    
    # Output the relevant summary
    return f"Here's a relevant summary for your query: \n{relevant_summary}"


In [35]:
import json

def chatbot_json(query):
    result = {
        "query": query,
        "relevant_summary": query_news(query, index)
    }
    return json.dumps(result, indent=2)

# Example usage
print(chatbot_json("What’s new in AI today?"))


{
  "query": "What\u2019s new in AI today?",
  "relevant_summary": "AI regulations are evolving rapidly in the US, with new guidelines being introduced for the tech industry."
}


In [36]:
summaries = [
    "AI regulations are evolving rapidly in the US, with new guidelines being introduced for the tech industry.",
    "Tech companies are racing to develop quantum computing, and major breakthroughs are expected in 2025.",
    "The latest smartphone from XYZ brand features groundbreaking AI capabilities and a new design."
    # Add more summaries as needed
]



In [37]:
# Example queries
query1 = "What is the latest news on AI?"
query2 = "Summarize today's technology news"

print(chatbot(query1))
print(chatbot(query2))


Here's a relevant summary for your query: 
AI regulations are evolving rapidly in the US, with new guidelines being introduced for the tech industry.
Here's a relevant summary for your query: 
AI regulations are evolving rapidly in the US, with new guidelines being introduced for the tech industry.


## 🔽 Step 8: Start a Conversational Chat with the Assistant


In [38]:
def start_chat():
    print("GenAI News Digest Assistant is ready! Ask me anything about the latest news.")
    while True:
        query = input("Your question: ")
        if query.lower() == "exit":
            break
        print(chatbot(query))

start_chat()


GenAI News Digest Assistant is ready! Ask me anything about the latest news.


Your question:  What's happening in the tech industry today?


Here's a relevant summary for your query: 
AI regulations are evolving rapidly in the US, with new guidelines being introduced for the tech industry.


Your question:  exit


## 🧠 Key Learnings, Limitations & Future Enhancements

### ✅ What I Learned

This capstone project provided practical exposure to applying Generative AI to address the challenge of information overload in the modern digital age. Key learnings include:

- Leveraged **Retrieval-Augmented Generation (RAG)** using FAISS for semantic search on news content.
- Applied **LLM-based summarization** using pretrained models from Hugging Face to condense real-time news articles.
- Built a lightweight **chatbot interface** to enable natural language interaction powered by vector search and generative responses.

### ⚠️ Known Limitations

- Summaries rely on the quality and recency of the source articles, which may introduce **bias or outdated information**.
- The summarization outputs are **not grounded or verified** unless explicitly guided with fact-checking logic.
- The current system lacks **user personalization** and **feedback adaptation** capabilities.

### 🚀 Opportunities for Future Work

- Integrate **real-time web scraping with scheduling** to update the article base dynamically.
- Enhance summarization accuracy and control using **fine-tuned or instruction-based models**.
- Introduce **user feedback mechanisms** (e.g., "Was this helpful?") and adapt responses using **structured JSON outputs**.
- Explore adding **multi-turn conversation flow** and **agent-style recommendations** to improve interactivity.

---

This wraps up the GenAI News Digest Assistant—a practical application of generative AI for smarter news consumption.
